# HoloSyn Visual Interface (Gradio) — Integrates Your Distilled TorchScript Model

This Colab notebook builds a **visual UI** to:
- Load your distilled model: `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- Load normalization: `/mnt/data/student_norm_hf.json`
- Optionally load your archive: `/mnt/data/Archive.zip` and browse files
- Compute modality-specific features (text/audio/image/video/haptics)
- Run inference → **valence/arousal/calm/trust**
- Visualize:
  - meters + time series
  - optional **two-peer synchrony** (A/B)
- Export a JSON session log

**Privacy note:** Everything runs locally in the notebook runtime.

---

## Inputs expected
- `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- `/mnt/data/student_norm_hf.json`
- Optional: `/mnt/data/Archive.zip`


In [1]:
#@title 0) Install deps
#@title 0) Install deps
!pip -q install -U gradio numpy "pandas==2.2.2" pillow opencv-python soundfile librosa ffmpeg-python sentence-transformers transformers
!pip -q install -U torch torchvision torchaudio pyarrow
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import librosa
import soundfile as sf
import cv2

import gradio as gr
print("✅ Installed")

✅ Installed


In [2]:
#@title 1) Paths + load model
MODEL_PATH = "/content/student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "/content/student_norm_hf.json"
ARCHIVE_ZIP = "/content/Archive.zip"

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model loaded")
print("Feature dims:", len(NUMERIC_COLS))

✅ Model loaded
Feature dims: 789


In [3]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))


Archive present: True
audio 49
video 52
image 304
text 49
haptics 92
other 25


In [4]:
#@title 3) Feature extraction (must align with training feature schema)
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    # Return dict of features; missing features are 0.0.
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        preview = s[:1000]
    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector in the exact order expected by the student
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            # embeddings columns (clip_*, w2v_*) are not computed here; keep 0 unless you add embed models
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    # [valence, arousal, calm, trust]
    return y

In [5]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

## 5) Gradio UI

Two panels:
- **Single input inference** (pick modality + source)
- **Two-peer synchrony**: run A & B and compute cosine similarity on `[valence, arousal, calm, trust]`


In [6]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        # preview might be PIL Image
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def infer_pair(modalityA, fileA, textA, modalityB, fileB, textB):
    prevA, vA, aA, cA, tA, featsA = infer_one(modalityA, fileA, textA)
    prevB, vB, aB, cB, tB, featsB = infer_one(modalityB, fileB, textB)
    eA = np.array([vA,aA,cA,tA], dtype=np.float32)
    eB = np.array([vB,aB,cB,tB], dtype=np.float32)
    sync = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))
    return prevA, prevB, sync

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")
        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame) OR Text snippet", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)
        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def render(preview_obj, v,a,c,t, feats):
            # If preview is an image, show it; else show text in preview_txt
            if isinstance(preview_obj, Image.Image):
                return preview_obj, "", v,a,c,t, feats
            else:
                # show placeholder image blank
                return None, str(preview_obj), v,a,c,t, feats

        run_btn.click(
            fn=lambda m,f,txt: infer_one(m,f,txt),
            inputs=[modality,file_dd,free_text],
            outputs=[preview_txt,val,aro,calm,trust,feats_json],
        ).then(
            fn=lambda prev_txt, v,a,c,t, feats: render(prev_txt, v,a,c,t, feats),
            inputs=[preview_txt,val,aro,calm,trust,feats_json],
            outputs=[preview,preview_txt,val,aro,calm,trust,feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")
        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")
        run_pair = gr.Button("Run pair + synchrony")
        with gr.Row():
            prevA = gr.Image(label="Preview A", type="pil")
            prevB = gr.Image(label="Preview B", type="pil")
        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        run_pair.click(
            fn=infer_pair,
            inputs=[modalityA,fileA,textA, modalityB,fileB,textB],
            outputs=[prevA,prevB,sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e33eb367ada0c904d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        # --- FIX: Unified handler to safely route text vs images ---
        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        # --- FIX: Provide both Image and Text preview blocks for safely rendering dynamic modalities ---
        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        # --- FIX: Unified handler to safely route pairs ---
        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40b7337f48b3c393c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

Loading text embedding model (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and Norm loaded
Feature dims: 789


In [9]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer
import transformers
import huggingface_hub

print(f"Transformers version: {transformers.__version__}")
print(f"Hugging Face Hub version: {huggingface_hub.__version__}")

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

Transformers version: 5.2.0
Hugging Face Hub version: 1.5.0
Loading text embedding model (this may take a moment)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and Norm loaded
Feature dims: 789


In [10]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

Archive present: True
audio 49
video 52
image 304
text 49
haptics 92
other 25


In [11]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [12]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [55]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d829a0ebc88c43a711.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [56]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

Archive present: True
audio 49
video 52
image 304
text 49
haptics 117
other 0


In [57]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [58]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [59]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://36f42234f250d18a2f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
!pip install cirq qsimcirq torch torchvision numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
import cirq
import qsimcirq
import json
import numpy as np

In [19]:
# Load your multimodal normalization statistics
with open('student_norm_hf.json', 'r') as f:
    norm_stats = json.load(f)

# Extract means and standard deviations for PyTorch preprocessing
mu = torch.tensor(norm_stats['mu'], dtype=torch.float32)
sd = torch.tensor(norm_stats['sd'], dtype=torch.float32)

def normalize_features(features):
    # Avoid division by zero for any zero-variance features
    return (features - mu) / (sd + 1e-7)

In [20]:
class QSimLayer(nn.Module):
    def __init__(self, num_qubits):
        super(QSimLayer, self).__init__()
        self.num_qubits = num_qubits
        self.qubits = cirq.GridQubit.rect(1, num_qubits)
        self.simulator = qsimcirq.QSimSimulator()

        # Trainable PyTorch parameters for the quantum circuit
        self.theta = nn.Parameter(torch.rand(num_qubits) * np.pi)

    def forward(self, x):
        batch_size = x.size(0)
        out = torch.zeros(batch_size, self.num_qubits, device=x.device)

        # Iterating through the batch to simulate the quantum state
        for i in range(batch_size):
            circuit = cirq.Circuit()

            # 1. Feature Encoding: Mapping continuous variables (e.g., w2v) to Rx gates
            for j in range(self.num_qubits):
                feature_val = x[i, j].item()
                circuit.append(cirq.rx(feature_val)(self.qubits[j]))

            # 2. Trainable Variational Entanglement Layer
            for j in range(self.num_qubits - 1):
                circuit.append(cirq.CNOT(self.qubits[j], self.qubits[j+1]))
            for j in range(self.num_qubits):
                circuit.append(cirq.ry(self.theta[j].item())(self.qubits[j]))

            # 3. Simulate with qsimcirq for state-vector processing
            result = self.simulator.simulate(circuit)
            state_vector = result.state_vector()

            # Extract probabilities and project back to a real tensor
            probs = np.abs(state_vector)**2
            out[i] = torch.tensor(probs[:self.num_qubits], dtype=torch.float32)

        return out

In [21]:
# 1. Load the Teacher Model (e.g., your open source model or holosyn_heads)
teacher_model = torch.jit.load('holosyn_heads.torchscript.pt')
teacher_model.eval() # Freeze the teacher

# 2. Load the Student Model
student_model = torch.jit.load('student_distilled_heads_hf.torchscript.pt')
student_model.train()

# 3. Wrap student with the Quantum Layer
class QuantumDistilledStudent(nn.Module):
    def __init__(self, base_student, num_q_features=8):
        super(QuantumDistilledStudent, self).__init__()
        self.num_q_features = num_q_features
        # Routing a subset of features (like w2v embeddings) through the quantum layer
        self.quantum_layer = QSimLayer(num_qubits=num_q_features)
        self.base_student = base_student

    def forward(self, x):
        # Pass the first 'n' features through the VQC
        q_out = self.quantum_layer(x[:, :self.num_q_features])

        # Re-inject quantum features into the main tensor
        x_modified = x.clone()
        x_modified[:, :self.num_q_features] = q_out

        return self.base_student(x_modified)

model = QuantumDistilledStudent(student_model)

In [22]:
def distillation_loss(student_logits, teacher_logits, labels, T=2.0, alpha=0.5):
    """
    Combines KL Divergence from the teacher and Standard Loss from the labels.
    """
    hard_loss = F.cross_entropy(student_logits, labels)
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T * T)

    return alpha * hard_loss + (1. - alpha) * soft_loss

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Pseudo-training execution
# for batch in dataloader:
#     features, labels = batch
#     features = normalize_features(features)
#
#     with torch.no_grad():
#         teacher_preds = teacher_model(features)
#
#     student_preds = model(features)
#     loss = distillation_loss(student_preds, teacher_preds, labels)
#
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

In [23]:
class QuantumAutogradFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inputs, weights, simulator, qubits):
        """
        inputs: Input features mapped to the circuit.
        weights: Trainable parameters (theta).
        simulator: The qsimcirq instance.
        qubits: The grid of Cirq qubits.
        """
        ctx.save_for_backward(inputs, weights)
        ctx.simulator = simulator
        ctx.qubits = qubits

        batch_size = inputs.size(0)
        num_qubits = len(qubits)
        out = torch.zeros(batch_size, num_qubits, device=inputs.device)

        # Forward pass execution
        for i in range(batch_size):
            circuit = cirq.Circuit()

            # Encode inputs
            for j in range(num_qubits):
                circuit.append(cirq.rx(inputs[i, j].item())(qubits[j]))

            # Apply weights (trainable Entanglement Layer)
            for j in range(num_qubits - 1):
                circuit.append(cirq.CNOT(qubits[j], qubits[j+1]))
            for j in range(num_qubits):
                circuit.append(cirq.ry(weights[j].item())(qubits[j]))

            result = simulator.simulate(circuit)
            probs = np.abs(result.state_vector())**2
            out[i] = torch.tensor(probs[:num_qubits], dtype=torch.float32)

        return out

    @staticmethod
    def backward(ctx, grad_output):
        """
        Calculates gradients using the Parameter-Shift Rule.
        """
        inputs, weights = ctx.saved_tensors
        simulator = ctx.simulator
        qubits = ctx.qubits

        batch_size = inputs.size(0)
        num_qubits = len(qubits)

        # We only need gradients for the trainable weights, not the simulator objects
        grad_weights = torch.zeros_like(weights)

        # Shift magnitude
        shift = np.pi / 2.0

        # Calculate gradients for each parameter
        for p_idx in range(len(weights)):
            # Create shifted parameter tensors
            weight_plus = weights.clone()
            weight_minus = weights.clone()

            weight_plus[p_idx] += shift
            weight_minus[p_idx] -= shift

            # We must execute the batch for both the + shift and - shift
            out_plus = torch.zeros(batch_size, num_qubits, device=inputs.device)
            out_minus = torch.zeros(batch_size, num_qubits, device=inputs.device)

            for i in range(batch_size):
                # Build and simulate circuit for + shift
                circ_plus = cirq.Circuit()
                for j in range(num_qubits): circ_plus.append(cirq.rx(inputs[i, j].item())(qubits[j]))
                for j in range(num_qubits - 1): circ_plus.append(cirq.CNOT(qubits[j], qubits[j+1]))
                for j in range(num_qubits): circ_plus.append(cirq.ry(weight_plus[j].item())(qubits[j]))

                res_plus = simulator.simulate(circ_plus)
                out_plus[i] = torch.tensor(np.abs(res_plus.state_vector())**2[:num_qubits])

                # Build and simulate circuit for - shift
                circ_minus = cirq.Circuit()
                for j in range(num_qubits): circ_minus.append(cirq.rx(inputs[i, j].item())(qubits[j]))
                for j in range(num_qubits - 1): circ_minus.append(cirq.CNOT(qubits[j], qubits[j+1]))
                for j in range(num_qubits): circ_minus.append(cirq.ry(weight_minus[j].item())(qubits[j]))

                res_minus = simulator.simulate(circ_minus)
                out_minus[i] = torch.tensor(np.abs(res_minus.state_vector())**2[:num_qubits])

            # Apply the parameter shift rule formula
            gradients = 0.5 * (out_plus - out_minus)

            # Chain rule: multiply the quantum gradients by the incoming gradients from the student model
            grad_weights[p_idx] = torch.sum(gradients * grad_output)

        # Return gradients for each input to `forward`. None for inputs that don't require gradients.
        # Gradients for: (inputs, weights, simulator, qubits)
        return None, grad_weights, None, None

<>:77: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:86: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:77: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:86: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
/tmp/ipython-input-1035806446.py:77: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
  out_plus[i] = torch.tensor(np.abs(res_plus.state_vector())**2[:num_qubits])
/tmp/ipython-input-1035806446.py:86: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
  out_minus[i] = torch.tensor(np.abs(res_minus.state_vector())**2[:num_qubits])


In [24]:
class DifferentiableQSimLayer(nn.Module):
    def __init__(self, num_qubits):
        super(DifferentiableQSimLayer, self).__init__()
        self.num_qubits = num_qubits
        self.qubits = cirq.GridQubit.rect(1, num_qubits)
        self.simulator = qsimcirq.QSimSimulator()

        # Initialize trainable parameters
        self.theta = nn.Parameter(torch.rand(num_qubits) * np.pi)

    def forward(self, x):
        # Call the custom Autograd function
        return QuantumAutogradFunction.apply(x, self.theta, self.simulator, self.qubits)

In [25]:
!pip install brian2 requests

import torch
from brian2 import *
import requests
import time
import numpy as np

# Set Brian2 to use numpy for standard execution, or cython for performance later
prefs.codegen.target = 'numpy'

In [26]:
def fetch_starlink_telemetry():
    """
    Fetches real-time telemetry from the local Starlink Dish API.
    Returns a normalized tensor of features: [ping_drop_rate, ping_latency, downlink_bps, uplink_bps]
    """
    try:
        # Standard local IP for Starlink Dishy
        response = requests.get("http://192.168.100.1/api/v1/status", timeout=2)
        data = response.json()

        # Extract raw features (mocking the structure if offline)
        drop_rate = data.get('popPingDropRate', 0.0)
        latency = data.get('popPingLatencyMs', 40.0)
        downlink = data.get('downlinkThroughputBps', 1000000)
        uplink = data.get('uplinkThroughputBps', 500000)

    except requests.exceptions.RequestException:
        # Fallback simulated data for testing the pipeline when offline
        drop_rate = np.random.uniform(0.0, 0.05)
        latency = np.random.uniform(20.0, 80.0)
        downlink = np.random.uniform(1e6, 50e6)
        uplink = np.random.uniform(5e5, 10e6)

    # Heuristic normalization to map into a reasonable range for quantum rotation gates [-pi, pi]
    features = [
        drop_rate * 10,                 # Scale drop rate
        (latency - 40) / 20,            # Normalize latency around 40ms
        np.log10(downlink + 1) / 8,     # Log scale for bps
        np.log10(uplink + 1) / 7
    ]

    # Return as a batch of size 1
    return torch.tensor([features], dtype=torch.float32)

In [27]:
def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope() # Clear the Brian2 registry for a fresh run

    # 1. The LIF Neuron Equation
    tau = 10*ms
    eqs = '''
    dv/dt = (1 - v) / tau : 1 (unless refractory)
    '''

    # 2. Network Layers
    # Input layer driven by Poisson rates derived from the Quantum Layer
    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)

    # Hidden processing layer
    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0', refractory=2*ms, method='exact')

    # 3. Synapses (Connections)
    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5) # 50% sparsity
    S.w = 'rand() * 0.2' # Random initial weights

    # 4. Monitors for tracking the engine's output
    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

In [28]:
def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope() # Clear the Brian2 registry for a fresh run

    # 1. The LIF Neuron Equation
    tau = 10*ms
    eqs = '''
    dv/dt = (1 - v) / tau : 1 (unless refractory)
    '''

    # 2. Network Layers
    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)

    # Explicitly pass the namespace here to fix the KeyError
    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0',
                           refractory=2*ms, method='exact', namespace={'tau': tau})

    # 3. Synapses (Connections)
    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5)
    S.w = 'rand() * 0.2'

    # 4. Monitors
    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

In [29]:
class SpikingSurrogateAutograd(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_rates, synaptic_weights, snn_network, S_synapse, monitor):
        """
        Forward pass: Sync PyTorch weights to Brian2, run simulation, return spike counts.
        """
        # 1. Sync PyTorch weights -> Brian2 SNN
        # S_synapse.w expects a flat numpy array
        S_synapse.w = synaptic_weights.detach().cpu().numpy()

        # 2. Set input rates and run
        # Assuming input_rates is mapped to a PoissonGroup (snn_network.objects[0])
        poisson_group = next(obj for obj in snn_network.objects if isinstance(obj, PoissonGroup))
        poisson_group.rates = input_rates.detach().cpu().numpy() * Hz

        # Reset and run the network
        snn_network.restore('initial_state') # Ensure we start fresh each batch
        snn_network.run(100*ms)

        # 3. Extract outputs (spike counts per neuron)
        spike_counts = torch.tensor(monitor.count[:], dtype=torch.float32, device=input_rates.device)

        # Save context for the backward pass
        ctx.save_for_backward(input_rates, synaptic_weights, spike_counts)
        ctx.S_synapse = S_synapse

        return spike_counts

    @staticmethod
    def backward(ctx, grad_output):
        """
        Backward pass: Approximate the gradients for the synaptic weights.
        """
        input_rates, synaptic_weights, spike_counts = ctx.saved_tensors
        S_synapse = ctx.S_synapse

        # Surrogate Gradient Approximation
        # Since exact Backprop-Through-Time (BPTT) is complex in Brian2,
        # we approximate the gradient of the weights based on the pre/post firing rates.
        # Simple Hebbian-like pseudo-gradient: grad_w ~ grad_out * pre_activity

        # Extract pre-synaptic and post-synaptic indices
        sources = S_synapse.i[:]
        targets = S_synapse.j[:]

        grad_weights = torch.zeros_like(synaptic_weights)

        # Accumulate gradients for each synapse based on the error at the target neuron
        # and the activity of the source neuron (input rate).
        for idx in range(len(sources)):
            pre_neuron = sources[idx]
            post_neuron = targets[idx]

            # The surrogate derivative: error * input_activity
            # Adding a small constant to ensure non-zero flow
            surrogate_derivative = grad_output[post_neuron] * (input_rates[pre_neuron] + 1e-4)
            grad_weights[idx] = surrogate_derivative

        return None, grad_weights, None, None, None

In [30]:
class Brian2PyTorchWrapper(nn.Module):
    def __init__(self, snn_network, P_input, S_synapse, monitor):
        super(Brian2PyTorchWrapper, self).__init__()
        self.snn_network = snn_network
        self.P_input = P_input
        self.S_synapse = S_synapse
        self.monitor = monitor

        # Store the initial state of the network so we can reset it every forward pass
        self.snn_network.store('initial_state')

        # Register the Brian2 weights as a trainable PyTorch parameter
        initial_weights = torch.tensor(S_synapse.w[:], dtype=torch.float32)
        self.synaptic_weights = nn.Parameter(initial_weights)

    def forward(self, x):
        return SpikingSurrogateAutograd.apply(
            x,
            self.synaptic_weights,
            self.snn_network,
            self.S_synapse,  # Fixed the double 'self' typo here
            self.monitor
        )

In [31]:
def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope() # Clear the Brian2 registry for a fresh run

    # 1. The LIF Neuron Equation
    # We bake the 10*ms time constant directly into the string to avoid ALL namespace scope issues
    eqs = '''
    dv/dt = (1 - v) / (10*ms) : 1 (unless refractory)
    '''

    # 2. Network Layers
    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)

    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0', refractory=2*ms, method='exact')

    # 3. Synapses (Connections)
    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5)
    S.w = 'rand() * 0.2'

    # 4. Monitors
    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

In [32]:
num_telemetry_features = 4
snn_network, snn_input, snn_monitor = setup_brian2_snn(num_input_neurons=num_telemetry_features)

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cirq
import qsimcirq
from brian2 import *
import numpy as np
import time

# ---------------------------------------------------------
# 1. SNN Setup (Namespace and Set-Indexing Fixed)
# ---------------------------------------------------------
prefs.codegen.target = 'numpy'

def setup_brian2_snn(num_input_neurons, num_hidden_neurons=20):
    start_scope()

    # Time constant is hardcoded into the string to avoid namespace KeyError
    eqs = '''
    dv/dt = (1 - v) / (10*ms) : 1 (unless refractory)
    '''

    P_input = PoissonGroup(num_input_neurons, rates=0*Hz)
    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0', refractory=2*ms, method='exact')

    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w')
    S.connect(p=0.5)
    S.w = 'rand() * 0.2'

    spike_monitor = SpikeMonitor(G_hidden)
    state_monitor = StateMonitor(G_hidden, 'v', record=True)

    network = Network(P_input, G_hidden, S, spike_monitor, state_monitor)
    return network, P_input, spike_monitor

# ---------------------------------------------------------
# 2. PyTorch to Brian2 Autograd Bridge
# ---------------------------------------------------------
class SpikingSurrogateAutograd(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_rates, synaptic_weights, snn_network, S_synapse, monitor):
        S_synapse.w = synaptic_weights.detach().cpu().numpy()

        poisson_group = next(obj for obj in snn_network.objects if isinstance(obj, PoissonGroup))
        poisson_group.rates = input_rates.detach().cpu().numpy() * Hz

        snn_network.restore('initial_state')
        snn_network.run(100*ms)

        spike_counts = torch.tensor(monitor.count[:], dtype=torch.float32, device=input_rates.device)

        ctx.save_for_backward(input_rates, synaptic_weights, spike_counts)
        ctx.S_synapse = S_synapse
        return spike_counts

    @staticmethod
    def backward(ctx, grad_output):
        input_rates, synaptic_weights, _ = ctx.saved_tensors
        S_synapse = ctx.S_synapse

        sources = S_synapse.i[:]
        targets = S_synapse.j[:]

        grad_weights = torch.zeros_like(synaptic_weights)
        for idx in range(len(sources)):
            pre_neuron = sources[idx]
            post_neuron = targets[idx]
            # Surrogate derivative calculation
            grad_weights[idx] = grad_output[post_neuron] * (input_rates[pre_neuron] + 1e-4)

        return None, grad_weights, None, None, None

class Brian2PyTorchWrapper(nn.Module):
    def __init__(self, snn_network, P_input, S_synapse, monitor):
        super(Brian2PyTorchWrapper, self).__init__()
        self.snn_network = snn_network
        self.P_input = P_input
        self.S_synapse = S_synapse
        self.monitor = monitor

        self.snn_network.store('initial_state')

        initial_weights = torch.tensor(S_synapse.w[:], dtype=torch.float32)
        self.synaptic_weights = nn.Parameter(initial_weights)

    def forward(self, x):
        return SpikingSurrogateAutograd.apply(
            x, self.synaptic_weights, self.snn_network, self.S_synapse, self.monitor
        )

# ---------------------------------------------------------
# 3. Execution & Training Pipeline
# ---------------------------------------------------------
# Assume q_layer is your DifferentiableQSimLayer defined earlier
# Assume student_model and teacher_model are loaded
# Assume fetch_starlink_telemetry() and distillation_loss() are defined

num_telemetry_features = 4
snn_net, snn_inp, snn_mon = setup_brian2_snn(num_input_neurons=num_telemetry_features)

# Dynamically find the Synapses object to avoid the 'set' TypeError
snn_synapse_obj = next(obj for obj in snn_net.objects if isinstance(obj, Synapses))
snn_module = Brian2PyTorchWrapper(snn_net, snn_inp, snn_synapse_obj, snn_mon)

# Add your optimizer and training loop here...

In [ ]:
!pip install gradio sentence-transformers

import gradio as gr
import torch

# For generating mock 768-dimensional embeddings to represent your LLM/W2V text input
# In production, replace this with your actual Gemini API call or local embedding model
class LLMEmbeddingMock:
    def encode(self, text):
        # Returns a normalized 768-dimensional tensor
        return torch.rand(1, 768)

llm_embedder = LLMEmbeddingMock()

def process_multimodal_forward_pass(text_input, snn_module, q_layer, student_model):
    """
    Executes the full forward pass: Telemetry -> Quantum -> SNN + LLM Text -> Student
    """
    # 1. Fetch & Quantum Encode Telemetry
    telemetry_tensor = fetch_starlink_telemetry()
    with torch.no_grad():
        q_probs = q_layer(telemetry_tensor)
        q_rates = q_probs.squeeze() * 100.0

    # 2. SNN Forward Pass (Output shape: [1, num_hidden_neurons])
    spike_counts = snn_module(q_rates).unsqueeze(0)

    # 3. LLM Text Embedding (Output shape: [1, 768])
    text_embeddings = llm_embedder.encode(text_input)

    # 4. The Concatenation Bridge
    # Merges the spatial SNN spikes with the dense LLM embeddings.
    # Resulting shape: [1, num_hidden_neurons + 768]
    combined_features = torch.cat([spike_counts, text_embeddings], dim=1)

    # Note: Ensure your student_model's input layer is sized to accept
    # (num_hidden_neurons + 768) features to match this concatenated tensor!

    # 5. Final Student Prediction
    with torch.no_grad():
        student_logits = student_model(combined_features)

    return spike_counts, student_logits

In [35]:
def engine_ui_inference(uploaded_file, text_prompt):
    """
    The function triggered when you click 'Run Engine' in Gradio.
    """
    # Handle optional file upload
    file_content = ""
    if uploaded_file is not None:
        try:
            with open(uploaded_file.name, 'r') as f:
                file_content = f.read()
        except Exception as e:
            file_content = f"Error reading file: {e}"

    # Combine uploaded text with the manual prompt
    full_text_input = f"{text_prompt}\n\nContext from file: {file_content[:500]}..."

    # Run the Wanalytics pipeline
    try:
        spike_counts, logits = process_multimodal_forward_pass(
            full_text_input, snn_module, q_layer, student_model
        )

        # Format the output for the UI
        spikes_out = int(spike_counts.sum().item())
        predicted_class = torch.argmax(logits, dim=1).item()

        results = (
            f"🟢 Pipeline Execution Successful\n"
            f"-----------------------------------\n"
            f"Starlink -> Quantum SNN Spikes Fired: {spikes_out}\n"
            f"Student Model Raw Logits: {logits.numpy().tolist()}\n"
            f"Predicted Output Class: {predicted_class}"
        )
        return results

    except Exception as e:
        return f"🔴 Engine Error: {str(e)}"

# Build the Gradio Blocks Layout
with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:
    gr.Markdown("# Wanalytics Tech | Multimodal Analytical Engine")
    gr.Markdown("Inject Starlink telemetry via Quantum-SNN and concatenate with LLM text embeddings.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 1. Input Modalities")
            ui_file = gr.File(label="Upload Context Data (TXT/JSON/CSV)")
            ui_text = gr.Textbox(lines=4, label="Gemini Text Prompt or Instruction", placeholder="Enter your text prompt here...")
            submit_btn = gr.Button("Run Distillation Engine", variant="primary")

        with gr.Column():
            gr.Markdown("### 2. Engine Output")
            ui_output = gr.Textbox(lines=10, label="System Logs & Predictions", interactive=False)

    # Wire the button to the inference function
    submit_btn.click(
        fn=engine_ui_inference,
        inputs=[ui_file, ui_text],
        outputs=ui_output
    )

# Launch the UI
# Set share=True if you want to access this via a public link on your phone/tablet
wanalytics_ui.launch(share=True, debug=True)

WARNING    /tmp/ipython-input-1944560916.py:40: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://46747ec6246442e9a5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5e33eb367ada0c904d.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://40b7337f48b3c393c7.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://bcb05860d32fb5b7e0.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://3c9d9afcb96a6b86ef.gradio.live
Killing tunnel 127.0.0.1:7864 <> https://46747ec6246442e9a5.gradio.live


In [36]:
import gradio as gr
import torch
import numpy as np
from brian2 import *

# ---------------------------------------------------------
# 1. Initialize Global Engine Components
# ---------------------------------------------------------
print("Initializing Wanalytics Quantum-SNN Engine...")
num_telemetry_features = 4

# Instantiate the Quantum Layer (using our DifferentiableQSimLayer)
qsim_vqc = DifferentiableQSimLayer(num_qubits=num_telemetry_features)

# Instantiate the Brian2 SNN Wrapper
snn_net, snn_inp, snn_mon = setup_brian2_snn(num_input_neurons=num_telemetry_features)
snn_synapse_obj = next(obj for obj in snn_net.objects if isinstance(obj, Synapses))
snn_module = Brian2PyTorchWrapper(snn_net, snn_inp, snn_synapse_obj, snn_mon)

# Load the distilled PyTorch student model
try:
    student_model = torch.jit.load('student_distilled_heads_hf.torchscript.pt')
    student_model.eval()
    print("Student model loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load student model. Using a mock linear layer for UI testing. ({e})")
    # Fallback to a mock model so the UI still runs if weights are missing in this directory
    student_model = torch.nn.Linear(num_hidden_neurons + 768, 2)

# Mock LLM Embedder for the text prompt (Returns 768-dim tensor)
class LLMEmbeddingMock:
    def encode(self, text):
        return torch.rand(1, 768)
llm_embedder = LLMEmbeddingMock()

# ---------------------------------------------------------
# 2. The Integrated Inference Function
# ---------------------------------------------------------
def run_quantum_snn_inference(file_upload, text_prompt):
    """
    Executes: Starlink Telemetry -> qsimcirq -> Brian2 -> Concatenation -> Student Model
    """
    try:
        # Step A: Get Starlink Telemetry (from your fetcher function)
        telemetry_tensor = fetch_starlink_telemetry()

        # Step B: Quantum Forward Pass
        with torch.no_grad(): # Inference mode
            q_probs = qsim_vqc(telemetry_tensor)
            q_rates = q_probs.squeeze() * 100.0 # Map to Hz

        # Step C: Brian2 SNN Forward Pass
        # Returns [1, num_hidden_neurons] tensor of spike counts
        spike_counts = snn_module(q_rates).unsqueeze(0)

        # Step D: Process Text / File Data
        file_context = ""
        if file_upload is not None:
            with open(file_upload.name, 'r') as f:
                file_context = f.read()[:500] # Limit context for embedding

        full_text = f"{text_prompt} | Context: {file_context}"
        text_embeddings = llm_embedder.encode(full_text)

        # Step E: Multimodal Concatenation
        # Combine Spikes (Spatial) + LLM Embeddings (Dense)
        combined_features = torch.cat([spike_counts, text_embeddings], dim=1)

        # Step F: Final Prediction through Distilled Heads
        with torch.no_grad():
            logits = student_model(combined_features)
            predicted_class = torch.argmax(logits, dim=1).item()
            confidence = torch.softmax(logits, dim=1)[0][predicted_class].item() * 100

        # Format Results
        total_spikes = int(spike_counts.sum().item())
        quantum_state_str = np.array2string(q_probs.numpy(), precision=3)

        return (
            f"✅ Engine Execution Successful\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"🛰️ Telemetry Ingested: {telemetry_tensor.numpy().round(3)}\n"
            f"⚛️ Quantum Probabilities: {quantum_state_str}\n"
            f"⚡ SNN Spikes Fired: {total_spikes}\n"
            f"🧠 Distilled Model Output Class: {predicted_class} (Confidence: {confidence:.1f}%)\n"
        )

    except Exception as e:
        return f"❌ Engine Failure: {str(e)}"

# ---------------------------------------------------------
# 3. Gradio Blocks Interface
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:
    gr.Markdown("# Wanalytics Tech | Quantum-Spiking Analytical Engine")
    gr.Markdown("Interactive inference dashboard for hybrid quantum, neuromorphic, and distilled LLM pipelines.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Data Ingestion")
            ui_text = gr.Textbox(lines=3, label="Text Prompt / Instructions", placeholder="Analyze the current telemetry state...")
            ui_file = gr.File(label="Upload Supplemental Data (JSON/TXT)")

            run_btn = gr.Button("Execute Quantum-SNN Forward Pass", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 2. Live Engine Telemetry")
            ui_output = gr.Textbox(lines=12, label="Execution Logs & Classifications", interactive=False)

    run_btn.click(
        fn=run_quantum_snn_inference,
        inputs=[ui_file, ui_text],
        outputs=ui_output
    )

# Launch the UI locally
wanalytics_ui.launch(share=True, debug=True)

WARNING    The object 'synapses_1' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipython-input-3940934465.py', line 26, in setup_brian2_snn
    S = Synapses(P_input, G_hidden, 'w : 1', on_pre='v_post += w') [brian2.core.base.unused_brian_object]
WARNING    /tmp/ipython-input-176461252.py:94: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Monochrome()) as wanalytics_ui:



Initializing Wanalytics Quantum-SNN Engine...
Student model loaded successfully.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6b6a01abbe2f863279.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


WARNING    The object 'poissongroup_1' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipython-input-3940934465.py', line 23, in setup_brian2_snn
    P_input = PoissonGroup(num_input_neurons, rates=0*Hz) [brian2.core.base.unused_brian_object]
WARNING    The object 'neurongroup_1' is getting deleted, but was never included in a network. This probably means that you did not store the object reference in a variable, or that the variable was not used to construct the network.
The object was created here (most recent call only):
  File '/tmp/ipython-input-3940934465.py', line 24, in setup_brian2_snn
    G_hidden = NeuronGroup(num_hidden_neurons, eqs, threshold='v>0.8', reset='v = 0', refractory=2*ms, method='exact') [brian2.core.base.unused_brian_object]


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://6b6a01abbe2f863279.gradio.live


In [37]:
import gradio as gr
import torch
import numpy as np
import time

# ---------------------------------------------------------
# 1. The Integrated Inference Function
# ---------------------------------------------------------
def run_quantum_snn_dashboard(text_prompt, file_upload, override_telemetry, drop_rate, latency):
    """
    Executes the full multimodal pipeline and formats it for a clean UI presentation.
    """
    try:
        start_time = time.time()

        # Step A: Telemetry Handling (Real or Mock via UI)
        if override_telemetry:
            # Use UI sliders if Starlink is offline or you want to manually test edge cases
            features = [drop_rate * 10, (latency - 40) / 20, np.log10(1000000)/8, np.log10(500000)/7]
            telemetry_tensor = torch.tensor([features], dtype=torch.float32)
        else:
            telemetry_tensor = fetch_starlink_telemetry()

        # Step B: Quantum Forward Pass
        with torch.no_grad():
            q_probs = qsim_vqc(telemetry_tensor)
            q_rates = q_probs.squeeze() * 100.0

        # Step C: Brian2 SNN Forward Pass
        spike_counts = snn_module(q_rates).unsqueeze(0)

        # Step D: Process Text Data
        file_context = ""
        if file_upload is not None:
            with open(file_upload.name, 'r') as f:
                file_context = f.read()[:500]

        full_text = f"{text_prompt} | {file_context}"
        text_embeddings = llm_embedder.encode(full_text)

        # Step E: Multimodal Concatenation & Final Prediction
        combined_features = torch.cat([spike_counts, text_embeddings], dim=1)

        with torch.no_grad():
            logits = student_model(combined_features)
            predicted_class = torch.argmax(logits, dim=1).item()
            confidence = torch.softmax(logits, dim=1)[0][predicted_class].item() * 100

        # Step F: UI Formatting
        total_spikes = int(spike_counts.sum().item())
        inference_time = round((time.time() - start_time) * 1000, 2)

        report = f"""
        ### 🟢 Engine Execution Successful ({inference_time} ms)

        **1. Starlink Telemetry Ingest**
        * Normalized Tensor: `{telemetry_tensor.numpy().round(3).tolist()}`

        **2. Quantum Entanglement Layer (qsimcirq)**
        * State Probabilities: `{np.array2string(q_probs.numpy(), precision=3)}`
        * Mapped Firing Rates: `{np.array2string(q_rates.numpy(), precision=1)} Hz`

        **3. Neuromorphic Spiking Network (Brian2)**
        * Total Network Spikes: **{total_spikes}**

        **4. Distilled Student Model (Multimodal)**
        * Predicted Class: **{predicted_class}**
        * Confidence: **{confidence:.2f}%**
        """

        raw_logs = f"Logits: {logits.numpy().tolist()}\nSpike Array: {spike_counts.numpy().tolist()}"
        return report, raw_logs

    except Exception as e:
        return f"### 🔴 Engine Failure\nError: {str(e)}", str(e)

# ---------------------------------------------------------
# 2. Advanced Gradio Blocks Interface
# ---------------------------------------------------------
# Changed the theme to gr.themes.Soft() to fix the AttributeError
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
    gr.Markdown(
        """
        # 🌌 Wanalytics Tech: Quantum-Spiking Engine
        **Multimodal Distillation Dashboard** integrating Starlink Telemetry, Cirq, Brian2, and LLM Embeddings.
        """
    )

    with gr.Tabs():
        with gr.TabItem("Live Inference"):
            with gr.Row():
                with gr.Column(scale=4):
                    gr.Markdown("### Modality Inputs")
                    text_input = gr.Textbox(lines=3, label="Text Prompt", placeholder="Enter analytical instructions...")
                    file_input = gr.File(label="Upload Context Data (JSON/TXT/CSV)")

                    with gr.Accordion("Manual Telemetry Override", open=False):
                        override_check = gr.Checkbox(label="Override Live Starlink Data", value=False)
                        drop_slider = gr.Slider(0.0, 1.0, value=0.05, label="Mock Ping Drop Rate")
                        lat_slider = gr.Slider(10.0, 200.0, value=40.0, label="Mock Latency (ms)")

                    run_btn = gr.Button("🚀 Run Quantum-SNN Pipeline", variant="primary")

                with gr.Column(scale=6):
                    gr.Markdown("### Engine Output")
                    main_output = gr.Markdown(label="Execution Report")

        with gr.TabItem("Raw Diagnostics"):
            raw_logs_output = gr.Textbox(lines=15, label="Raw Tensor Logs", interactive=False)

    run_btn.click(
        fn=run_quantum_snn_dashboard,
        inputs=[text_input, file_input, override_check, drop_slider, lat_slider],
        outputs=[main_output, raw_logs_output]
    )

print("Generating public Gradio link...")
wanalytics_dashboard.launch(share=True, debug=True)

WARNING    /tmp/ipython-input-3989220119.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:



Generating public Gradio link...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5360559ba79784b1b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://5360559ba79784b1b7.gradio.live


In [38]:
import gradio as gr
import torch
import torch.nn.functional as F
import numpy as np
import time

# ---------------------------------------------------------
# 1. Initialize Global Engine Components (Dual Models)
# ---------------------------------------------------------
print("Initializing Wanalytics Dual-Model Engine...")

# Load the SNN and Quantum Layers
num_telemetry_features = 4
qsim_vqc = DifferentiableQSimLayer(num_qubits=num_telemetry_features)
snn_net, snn_inp, snn_mon = setup_brian2_snn(num_input_neurons=num_telemetry_features)
snn_synapse_obj = next(obj for obj in snn_net.objects if isinstance(obj, Synapses))
snn_module = Brian2PyTorchWrapper(snn_net, snn_inp, snn_synapse_obj, snn_mon)

# Load Both PyTorch Models
try:
    # The heavy baseline teacher
    holosyn_teacher = torch.jit.load('holosyn_heads.torchscript.pt')
    holosyn_teacher.eval()

    # The lightweight quantum-distilled student
    student_model = torch.jit.load('student_distilled_heads_hf.torchscript.pt')
    student_model.eval()
    print("Both Teacher and Student models loaded successfully.")
except Exception as e:
    print(f"Model Load Error: {e}")

# Mock LLM Embedder
class LLMEmbeddingMock:
    def encode(self, text):
        return torch.rand(1, 768)
llm_embedder = LLMEmbeddingMock()

# ---------------------------------------------------------
# 2. Dual-Inference Distillation Function
# ---------------------------------------------------------
def run_dual_engine_dashboard(text_prompt, file_upload, override_telemetry, drop_rate, latency):
    try:
        start_time = time.time()

        # A: Starlink Telemetry
        if override_telemetry:
            features = [drop_rate * 10, (latency - 40) / 20, np.log10(1000000)/8, np.log10(500000)/7]
            telemetry_tensor = torch.tensor([features], dtype=torch.float32)
        else:
            telemetry_tensor = fetch_starlink_telemetry()

        # B: LLM Text Embedding (768 dimensions)
        file_context = ""
        if file_upload is not None:
            with open(file_upload.name, 'r') as f:
                file_context = f.read()[:500]
        text_embeddings = llm_embedder.encode(f"{text_prompt} | {file_context}")

        # ---------------------------------------------------------
        # PIPELINE 1: The Quantum-Distilled Student (789 dims)
        # ---------------------------------------------------------
        with torch.no_grad():
            q_probs = qsim_vqc(telemetry_tensor)
            q_rates = q_probs.squeeze() * 100.0

        spike_counts = snn_module(q_rates).unsqueeze(0) # [1, 20]
        padding = torch.zeros(1, 1, dtype=torch.float32, device=spike_counts.device) # [1, 1]

        # [20 SNN Spikes + 1 Padding + 768 Text] = 789
        student_features = torch.cat([spike_counts, padding, text_embeddings], dim=1)

        with torch.no_grad():
            student_logits = student_model(student_features)
            student_class = torch.argmax(student_logits, dim=1).item()
            student_conf = torch.softmax(student_logits, dim=1)[0][student_class].item() * 100

        # ---------------------------------------------------------
        # PIPELINE 2: The Holosyn Teacher Baseline (789 dims)
        # ---------------------------------------------------------
        # The teacher bypasses the SNN and expects standard normalized metadata
        # For the UI comparison, we map the raw telemetry directly into the first 4 slots
        # and pad the remaining 17 metadata slots with zeros to equal 21.
        teacher_metadata = torch.zeros(1, 21, dtype=torch.float32)
        teacher_metadata[0, :4] = telemetry_tensor[0]

        # [21 Standard Metadata + 768 Text] = 789
        teacher_features = torch.cat([teacher_metadata, text_embeddings], dim=1)

        with torch.no_grad():
            teacher_logits = holosyn_teacher(teacher_features)
            teacher_class = torch.argmax(teacher_logits, dim=1).item()
            teacher_conf = torch.softmax(teacher_logits, dim=1)[0][teacher_class].item() * 100

        # ---------------------------------------------------------
        # UI Formatting
        # ---------------------------------------------------------
        inference_time = round((time.time() - start_time) * 1000, 2)
        match_status = "✅ CONVERGED" if student_class == teacher_class else "⚠️ DIVERGED"

        report = f"""
        ### ⏱️ Dual Inference Completed ({inference_time} ms)

        #### 🧠 Holosyn Teacher Baseline (Classical)
        * Predicted Class: **{teacher_class}**
        * Confidence: **{teacher_conf:.2f}%**

        #### ⚛️ Quantum-Distilled Student (qsimcirq + Brian2)
        * Total SNN Spikes: **{int(spike_counts.sum().item())}**
        * Predicted Class: **{student_class}**
        * Confidence: **{student_conf:.2f}%**

        ---
        ### Distillation Status: {match_status}
        """

        return report

    except Exception as e:
        return f"### 🔴 Engine Failure\nError: {str(e)}"

# ---------------------------------------------------------
# 3. Gradio Interface (Dual Model Dashboard)
# ---------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
    gr.Markdown(
        """
        # 🌌 Wanalytics Tech: Holosyn vs. Quantum Distillation
        Real-time benchmarking of the heavy `holosyn` teacher against the hybrid SNN-Quantum student.
        """
    )

    with gr.Row():
        with gr.Column(scale=4):
            gr.Markdown("### Input Modalities")
            text_input = gr.Textbox(lines=3, label="Multimodal Text Prompt")
            file_input = gr.File(label="Context Data")

            with gr.Accordion("Telemetry Override", open=False):
                override_check = gr.Checkbox(label="Enable UI Sliders", value=False)
                drop_slider = gr.Slider(0.0, 1.0, value=0.05, label="Ping Drop Rate")
                lat_slider = gr.Slider(10.0, 200.0, value=40.0, label="Latency (ms)")

            run_btn = gr.Button("🚀 Execute Dual Inference", variant="primary")

        with gr.Column(scale=6):
            gr.Markdown("### ⚖️ Distillation Comparison")
            main_output = gr.Markdown(label="Benchmarking Report")

    run_btn.click(
        fn=run_dual_engine_dashboard,
        inputs=[text_input, file_input, override_check, drop_slider, lat_slider],
        outputs=main_output
    )

print("Generating public Gradio link...")
wanalytics_dashboard.launch(share=True, debug=True)

Initializing Wanalytics Dual-Model Engine...


WARNING    /tmp/ipython-input-486415270.py:124: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:



Both Teacher and Student models loaded successfully.
Generating public Gradio link...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3e43671c37e6082f2a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://3e43671c37e6082f2a.gradio.live


In [39]:
def setup_synchrony_snn(num_inputs=4):
    start_scope()
    eqs = "dv/dt = (1 - v) / (10*ms) : 1 (unless refractory)"

    # Dual Input Groups for Two-Peer Synchrony
    P_A = PoissonGroup(num_inputs, rates=0*Hz, name='Peer_A')
    P_B = PoissonGroup(num_inputs, rates=0*Hz, name='Peer_B')

    G_hidden = NeuronGroup(20, eqs, threshold='v>0.8', reset='v=0', refractory=2*ms, method='exact')

    # Connect both peers to the same hidden "Emotional Integration" layer
    S_A = Synapses(P_A, G_hidden, 'w:1', on_pre='v_post += w')
    S_B = Synapses(P_B, G_hidden, 'w:1', on_pre='v_post += w')
    S_A.connect(p=0.5); S_B.connect(p=0.5)
    S_A.w = S_B.w = 0.2

    # Monitor to detect coincident spikes (Synchrony)
    mon = SpikeMonitor(G_hidden)
    net = Network(P_A, P_B, G_hidden, S_A, S_B, mon)
    return net, P_A, P_B, mon

In [62]:
def run_emotional_interface(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # 1. Force 2D Input Shape: (batch_size=1, features=1)
        # Using [[float]] ensures qsimcirq receives a rank-2 tensor
        peer_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
        peer_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)

        # 2. Quantum Encoding & Peer Synchrony (Brian2)
        with torch.no_grad():
            q_probs_a = qsim_vqc(peer_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            if peer_mode == "Two-Peer":
                q_probs_b = qsim_vqc(peer_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz

            snn_net.restore('initial_state')
            snn_net.run(100*ms)
            # Synchrony is measured by coincident spikes in the hidden layer
            sync_score = float(snn_mon.num_spikes)

        # 3. Align 789 Features: [20 Spikes + 1 Pad + 768 Text]
        # Your model expects 21 metadata slots before the W2V embeddings
        text_emb = llm_embedder.encode(text_prompt)
        spike_metadata = torch.full((1, 20), sync_score, dtype=torch.float32)
        meta_padding = torch.zeros(1, 1, dtype=torch.float32)

        # Total Shape: (1, 789)
        combined_input = torch.cat([spike_metadata, meta_padding, text_emb], dim=1)

        # 4. Inference with Dimension Safety Check
        with torch.no_grad():
            logits = student_model(combined_input)

            # Check if model returns 2 classes (Index 0 and Index 1)
            if logits.dim() > 1 and logits.shape[1] > 1:
                res_class = torch.argmax(logits, dim=1).item()
                confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100
                status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
            else:
                # Handle single-output regression cases gracefully
                res_val = logits.item()
                status = "📈 ANALYZING"
                confidence = res_val * 100

        return f"### {status}\n**Synchrony Score:** {sync_score}\n**Confidence:** {confidence:.1f}%"

    except Exception as e:
        return f"### 🔴 Interface Error\nInput Shape: {combined_input.shape if 'combined_input' in locals() else 'Unknown'}\nError: {str(e)}"

# Define the missing UI Blocks and Components
with gr.Blocks(theme=gr.themes.Soft()) as emotional_ui:
    gr.Markdown("# Wanalytics Emotional Interface")

    with gr.Row():
        prompt = gr.Textbox(label="Text Prompt", placeholder="Enter emotional context...")
        mode = gr.Radio(["Single", "Two-Peer"], label="Mode", value="Single")

    with gr.Row():
        data_a = gr.Slider(0, 1, value=0.5, label="Peer A Intensity")
        data_b = gr.Slider(0, 1, value=0.5, label="Peer B Intensity", visible=False)

    run_btn = gr.Button("Run Analysis", variant="primary")
    output_display = gr.Markdown(label="Output Result")

    # Toggle Peer B slider visibility
    mode.change(lambda m: gr.update(visible=(m == "Two-Peer")), inputs=mode, outputs=data_b)

    run_btn.click(
        fn=run_emotional_interface,
        inputs=[prompt, mode, data_a, data_b],
        outputs=output_display
    )

emotional_ui.launch(share=True)

WARNING    /tmp/ipython-input-1912945684.py:54: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as emotional_ui:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as emotional_ui:



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2bc10b2d73f5bd01a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [42]:
def run_emotional_interface(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # 1. Ensure 2D shape for Quantum Input: (batch_size=1, features=1)
        # This prevents the 'out of bounds' error during qsimcirq processing
        peer_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
        peer_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)

        # 2. Quantum & SNN Forward Pass
        with torch.no_grad():
            q_probs_a = qsim_vqc(peer_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            if peer_mode == "Two-Peer":
                q_probs_b = qsim_vqc(peer_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz

            snn_net.restore('initial_state')
            snn_net.run(100*ms)
            sync_score = float(snn_mon.num_spikes)

        # 3. Feature Realignment for 789-dim Distilled Model
        # metadata(21) + w2v(768) = 789
        text_emb = llm_embedder.encode(text_prompt) # Expected: (1, 768)

        # We must create exactly 21 metadata features to match the TorchScript expected shape
        # Mapping SNN spikes to the first 20 slots, leaving 1 for padding
        spike_metadata = torch.full((1, 20), sync_score, dtype=torch.float32)
        meta_padding = torch.zeros(1, 1, dtype=torch.float32)

        # FINAL CONCATENATION: (1, 20) + (1, 1) + (1, 768) = (1, 789)
        combined_input = torch.cat([spike_metadata, meta_padding, text_emb], dim=1)

        # 4. Inference
        with torch.no_grad():
            logits = student_model(combined_input)
            # Ensure we don't index into logits if it's empty
            if logits.shape[0] > 0:
                res_class = torch.argmax(logits, dim=1).item()
                confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100
            else:
                return "Error: Model returned empty logits."

        status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
        return f"### {status}\n**Synchrony Score:** {sync_score}\n**Confidence:** {confidence:.1f}%"

    except Exception as e:
        return f"### 🔴 Interface Error\n{str(e)}"

In [43]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

# Load the local tokenizer (e.g., BERT or RoBERTa) to match your 768-dim Word2Vec slots
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_model = AutoModel.from_pretrained("bert-base-uncased")

def run_synth_interface(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # Step A: Tokenizer & Embedding Generation
        inputs = tokenizer(text_prompt, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = text_model(**inputs)
            # Pool to get a (1, 768) vector for the 'w2v' slots
            text_emb = outputs.last_hidden_state[:, 0, :]

        # Step B: Quantum & SNN Synchrony
        peer_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
        with torch.no_grad():
            q_probs_a = qsim_vqc(peer_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            if peer_mode == "Two-Peer":
                peer_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)
                q_probs_b = qsim_vqc(peer_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz

            snn_net.restore('initial_state')
            snn_net.run(100*ms)
            sync_score = float(snn_mon.num_spikes)

        # Step C: 789-Feature Alignment [20 Spikes + 1 Pad + 768 Text]
        spike_metadata = torch.full((1, 20), sync_score, dtype=torch.float32)
        meta_padding = torch.zeros(1, 1, dtype=torch.float32)
        combined_input = torch.cat([spike_metadata, meta_padding, text_emb], dim=1)

        # Step D: Model Synthesis
        with torch.no_grad():
            logits = student_model(combined_input)
            res_class = torch.argmax(logits, dim=1).item()
            confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100

        status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
        return f"### {status}\n**Synchrony Score:** {sync_score}\n**Confidence:** {confidence:.1f}%"

    except Exception as e:
        return f"### 🔴 Synthesis Error\n{str(e)}"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
def run_synth_interface(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # 1. Tokenizer & Embedding Generation
        inputs = tokenizer(text_prompt, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = text_model(**inputs)
            text_emb = outputs.last_hidden_state[:, 0, :] # Shape: (1, 768)

        # 2. Quantum & SNN Forward Pass (Brampton Research Logic)
        peer_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
        with torch.no_grad():
            q_probs_a = qsim_vqc(peer_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            if peer_mode == "Two-Peer":
                peer_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)
                q_probs_b = qsim_vqc(peer_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz

            snn_net.restore('initial_state')
            snn_net.run(100*ms)
            sync_score = float(snn_mon.num_spikes)

        # 3. 789-Feature Alignment
        # metadata(21) + w2v(768) = 789
        spike_metadata = torch.full((1, 20), sync_score, dtype=torch.float32)
        meta_padding = torch.zeros(1, 1, dtype=torch.float32)
        combined_input = torch.cat([spike_metadata, meta_padding, text_emb], dim=1)

        # 4. Model Output Handling (Fixes IndexError)
        with torch.no_grad():
            logits = student_model(combined_input)

            # Check if output is Classification [1, 2] or Regression [1, 1]
            if logits.dim() > 1 and logits.shape[1] > 1:
                res_class = torch.argmax(logits, dim=1).item()
                confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100
                status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
            else:
                # Handle single-value output safely
                score = logits.item()
                status = "📈 INTENSITY ANALYZED"
                confidence = min(max(score * 100, 0), 100) # Clamp 0-100%

        return f"### {status}\n**Synchrony Score:** {sync_score}\n**Synthesis Confidence:** {confidence:.1f}%"

    except Exception as e:
        return f"### 🔴 Synthesis Error\nInput Shape: {combined_input.shape if 'combined_input' in locals() else 'N/A'}\nError: {str(e)}"

In [46]:
def run_synth_interface(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # 1. Tokenizer & Embedding Generation (768-dim)
        inputs = tokenizer(text_prompt, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = text_model(**inputs)
            # Standard pooling for Word2Vec/Transformer slots
            text_emb = outputs.last_hidden_state[:, 0, :]

        # 2. Quantum & SNN Forward Pass (Brampton Synchrony Logic)
        # Ensure 2D input [[val]] for qsimcirq
        peer_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
        with torch.no_grad():
            q_probs_a = qsim_vqc(peer_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            if peer_mode == "Two-Peer":
                peer_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)
                q_probs_b = qsim_vqc(peer_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz

            snn_net.restore('initial_state')
            snn_net.run(100*ms)
            sync_score = float(snn_mon.num_spikes)

        # 3. 789-Feature Alignment [20 Spikes + 1 Pad + 768 Text]
        spike_metadata = torch.full((1, 20), sync_score, dtype=torch.float32)
        meta_padding = torch.zeros(1, 1, dtype=torch.float32)
        combined_input = torch.cat([spike_metadata, meta_padding, text_emb], dim=1)

        # 4. Robust Inference (Fixes index 1 out of bounds)
        with torch.no_grad():
            logits = student_model(combined_input)

            # Logic to handle both Classification [1, 2] and Regression [1, 1]
            if logits.dim() > 1 and logits.shape[1] > 1:
                res_class = torch.argmax(logits, dim=1).item()
                confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100
                status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
            else:
                # Fallback for single-output regression models
                score = logits.item()
                status = "📈 INTENSITY ANALYZED"
                confidence = min(max(score * 100, 0), 100) # Percentage mapping

        return f"### {status}\n**Synchrony Score:** {sync_score}\n**Synthesis Confidence:** {confidence:.1f}%"

    except Exception as e:
        return f"### 🔴 Synthesis Error\nInput Shape: {combined_input.shape if 'combined_input' in locals() else 'N/A'}\nError: {str(e)}"

In [47]:
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_ui:
    gr.Markdown("# 🌌 Wanalytics Tech | Quantum-Spiking Synth")
    gr.Markdown("Brampton Research Division: Multi-Peer Emotional Synchrony")

    with gr.Tabs():
        with gr.TabItem("Inference Engine"):
            with gr.Row():
                with gr.Column():
                    ui_text = gr.Textbox(label="Tokenizer Input", placeholder="Describe the emotional context...")
                    ui_mode = gr.Radio(["Single", "Two-Peer"], label="Peer Mode", value="Single")
                    ui_a = gr.Slider(0, 1, value=0.5, label="Peer A Intensity")
                    ui_b = gr.Slider(0, 1, value=0.5, label="Peer B Intensity", visible=False)
                    run_btn = gr.Button("🚀 Execute Synthesis", variant="primary")

                with gr.Column():
                    ui_output = gr.Markdown("### Result Status")

        with gr.TabItem("System Settings"):
            gr.Markdown("Model Tier: Paid (Gemini 3 Flash Web)")
            gr.Markdown("Location: Brampton, Ontario")

    # Dynamic UI Logic
    ui_mode.change(lambda m: gr.update(visible=(m == "Two-Peer")), inputs=ui_mode, outputs=ui_b)

    run_btn.click(
        fn=run_synth_interface,
        inputs=[ui_text, ui_mode, ui_a, ui_b],
        outputs=ui_output
    )

wanalytics_ui.launch(share=True)

WARNING    /tmp/ipython-input-2741116808.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_ui:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_ui:



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9518ffa73dfd21c8d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [48]:
import gradio as gr
import torch
from brian2 import *
import numpy as np

def run_unified_benchmarking(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # Step A: Tokenizer & Embedding Generation (768-dim)
        inputs = tokenizer(text_prompt, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = text_model(**inputs)
            text_emb = outputs.last_hidden_state[:, 0, :]

        # Step B: Quantum-Spiking Synchrony (Peer A & B)
        # Force 2D for qsimcirq [[val]]
        p_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
        with torch.no_grad():
            q_probs_a = qsim_vqc(p_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            if peer_mode == "Two-Peer":
                p_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)
                q_probs_b = qsim_vqc(p_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz

            snn_net.restore('initial_state')
            snn_net.run(100*ms)
            sync_score = float(snn_mon.num_spikes)

        # Step C: Feature Alignment [20 Spikes + 1 Pad + 768 Text] = 789
        spike_meta = torch.full((1, 20), sync_score, dtype=torch.float32)
        meta_pad = torch.zeros(1, 1, dtype=torch.float32)
        combined_input = torch.cat([spike_meta, meta_pad, text_emb], dim=1)

        # Step D: Dual Model Inference (Holosyn vs. Student)
        with torch.no_grad():
            # 1. Quantum-Distilled Student
            student_logits = student_model(combined_input)

            # 2. Holosyn Teacher Baseline
            # Note: Teacher expects same 789 shape for consistency
            teacher_logits = holosyn_teacher(combined_input)

        # Step E: Output Formatting & Error-Proof Indexing
        def process_logits(logits):
            if logits.shape[1] > 1: # Classification
                idx = torch.argmax(logits, dim=1).item()
                conf = torch.softmax(logits, dim=1)[0][idx].item() * 100
                label = "RESONANT" if idx == 1 else "DISSONANT"
            else: # Regression fallback
                val = logits.item()
                label = "INTENSITY"
                conf = val * 100
            return label, conf

        s_label, s_conf = process_logits(student_logits)
        t_label, t_conf = process_logits(teacher_logits)

        report = f"""
        ### 📊 Benchmarking Report: Wanalytics Tech
        * **Synchrony Score**: {sync_score} Spikes
        * **Peer Mode**: {peer_mode}

        | Model | Prediction | Confidence |
        | :--- | :--- | :--- |
        | **Holosyn Teacher** | {t_label} | {t_conf:.1f}% |
        | **Quantum Student** | {s_label} | {s_conf:.1f}% |
        """
        return report

    except Exception as e:
        return f"### 🔴 Synthesis Error\n{str(e)}"

In [49]:
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
    gr.Markdown("# 🌌 Wanalytics Unified Analytical Engine")
    gr.Markdown("Brampton Research: Quantum-Neuromorphic Synthesis")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🛠️ Input Config")
            ui_text = gr.Textbox(label="Prompt Tokenizer", lines=2)
            ui_mode = gr.Radio(["Single", "Two-Peer"], label="Synchrony Mode", value="Single")

            with gr.Row():
                ui_a = gr.Slider(0, 1, value=0.5, label="Peer A Intensity")
                ui_b = gr.Slider(0, 1, value=0.5, label="Peer B Intensity", visible=False)

            run_btn = gr.Button("🚀 Run Dual Inference", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### ⚖️ Synthesis Results")
            ui_report = gr.Markdown("Ready for inference...")

    # Dynamic peer visibility
    ui_mode.change(lambda m: gr.update(visible=(m == "Two-Peer")), inputs=ui_mode, outputs=ui_b)

    run_btn.click(
        fn=run_unified_benchmarking,
        inputs=[ui_text, ui_mode, ui_a, ui_b],
        outputs=ui_report
    )

# Launch with share=True for the .live link
wanalytics_dashboard.launch(share=True)

WARNING    /tmp/ipython-input-3713297603.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6262aee4a387485e5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [50]:
def run_distilled_synthesis(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # 1. Tokenizer & Embedding Generation
        inputs = tokenizer(text_prompt, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = text_model(**inputs)
            text_emb = outputs.last_hidden_state[:, 0, :] # Shape: (1, 768)

        # 2. Quantum & SNN Forward Pass
        p_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
        with torch.no_grad():
            q_probs_a = qsim_vqc(p_a_tensor)
            snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

            if peer_mode == "Two-Peer":
                p_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)
                q_probs_b = qsim_vqc(p_b_tensor)
                snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
            else:
                snn_inp_b.rates = 0 * Hz

            snn_net.restore('initial_state')
            snn_net.run(100*ms)
            sync_score = float(snn_mon.num_spikes)

        # 3. 789-Feature Alignment [20 Spikes + 1 Pad + 768 Text]
        # This matches the numeric_cols (21) + w2v (768) in student_norm_hf.json
        spike_meta = torch.full((1, 20), sync_score, dtype=torch.float32)
        meta_pad = torch.zeros(1, 1, dtype=torch.float32)
        combined_input = torch.cat([spike_meta, meta_pad, text_emb], dim=1)

        # 4. Dimension-Safe Inference
        with torch.no_grad():
            logits = student_model(combined_input)

            # Check for classification [1, 2] vs. regression [1, 1]
            if logits.dim() > 1 and logits.shape[1] > 1:
                res_class = torch.argmax(logits, dim=1).item()
                confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100
                status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
            else:
                # Regression fallback to prevent IndexError
                intensity = logits.item()
                status = "📈 INTENSITY ANALYZED"
                confidence = min(max(intensity * 100, 0), 100)

        return f"### {status}\n**Synchrony Score:** {sync_score}\n**Synthesis Confidence:** {confidence:.1f}%"

    except Exception as e:
        return f"### 🔴 Synthesis Error\n{str(e)}"

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_dashboard:
    gr.Markdown("# 🌌 Wanalytics Unified Distillation Engine")
    gr.Markdown("Brampton Research: Quantum-Neuromorphic Synthesis")

    with gr.Row():
        with gr.Column():
            ui_text = gr.Textbox(label="Tokenizer Input")
            ui_mode = gr.Radio(["Single", "Two-Peer"], label="Peer Mode", value="Single")
            ui_a = gr.Slider(0, 1, label="Peer A Intensity")
            ui_b = gr.Slider(0, 1, label="Peer B Intensity", visible=False)
            run_btn = gr.Button("🚀 Run Synthesis", variant="primary")

        with gr.Column():
            ui_report = gr.Markdown("Ready for distilled inference...")

    ui_mode.change(lambda m: gr.update(visible=(m == "Two-Peer")), inputs=ui_mode, outputs=ui_b)
    run_btn.click(fn=run_distilled_synthesis, inputs=[ui_text, ui_mode, ui_a, ui_b], outputs=ui_report)

wanalytics_dashboard.launch(share=True)

In [51]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModel
from brian2 import *
import numpy as np

# Load local Hugging Face tokenizer & embedding model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_model = AutoModel.from_pretrained("bert-base-uncased")

def encode_synth(text_prompt, peer_a_val, peer_b_val, peer_mode="Two-Peer"):
    """
    Ingests multimodal inputs and encodes them into a unified 789-dim tensor.
    """
    # 1. Text Synthesis (768 dimensions)
    inputs = tokenizer(text_prompt, return_tensors="pt", padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        text_emb = text_model(**inputs).last_hidden_state[:, 0, :]

    # 2. Quantum Encoding (Telemetry -> qsimcirq)
    # Strictly formatting inputs as 2D tensors [[float]] to satisfy quantum batch requirements
    p_a_tensor = torch.tensor([[float(peer_a_val)]], dtype=torch.float32)
    with torch.no_grad():
        q_probs_a = qsim_vqc(p_a_tensor)
        snn_inp_a.rates = q_probs_a.squeeze().numpy() * 100 * Hz

        if peer_mode == "Two-Peer":
            p_b_tensor = torch.tensor([[float(peer_b_val)]], dtype=torch.float32)
            q_probs_b = qsim_vqc(p_b_tensor)
            snn_inp_b.rates = q_probs_b.squeeze().numpy() * 100 * Hz
        else:
            snn_inp_b.rates = 0 * Hz

        # 3. Spiking Neuromorphic Encoding (Brian2)
        snn_net.restore('initial_state')
        snn_net.run(100*ms)
        sync_score = float(snn_mon.num_spikes)

    # 4. Multimodal Synthesis Assembly
    # Fills the 20 spike slots and 1 padding slot to align with the 768 text embeddings
    spike_meta = torch.full((1, 20), sync_score, dtype=torch.float32)
    meta_pad = torch.zeros(1, 1, dtype=torch.float32)

    # Final shape: [1, 789]
    synth_tensor = torch.cat([spike_meta, meta_pad, text_emb], dim=1)

    return synth_tensor, sync_score

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
def execute_quantum_synthesis(text_prompt, peer_mode, peer_a_val, peer_b_val):
    try:
        # Generate the encoded synthesis tensor safely
        synth_tensor, sync_score = encode_synth(text_prompt, peer_a_val, peer_b_val, peer_mode)

        # Pass the guaranteed [1, 789] tensor through the distilled student model
        with torch.no_grad():
            logits = student_model(synth_tensor)

            # Dimension-safe output parsing (prevents index out of bounds)
            if logits.dim() > 1 and logits.shape[1] > 1:
                res_class = torch.argmax(logits, dim=1).item()
                confidence = torch.softmax(logits, dim=1)[0][res_class].item() * 100
                status = "💞 RESONANT" if res_class == 1 else "💔 DISSONANT"
            else:
                score = logits.item()
                status = "📈 INTENSITY ANALYZED"
                confidence = min(max(score * 100, 0), 100)

        return f"### {status}\n**SNN Synchrony Spikes:** {sync_score}\n**Confidence:** {confidence:.1f}%\n\n*Encoded Tensor Shape: {list(synth_tensor.shape)}*"
    except Exception as e:
        return f"### 🔴 Synthesis Execution Error\n{str(e)}"

# Wanalytics Tech Gradio Interface
with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_ui:
    gr.Markdown("# 🌌 Wanalytics Tech | Quantum Encoded Synthesis")
    gr.Markdown("Award-nominated AI research architecture. Generates 789-dim multimodal tensors via qsimcirq and Brian2.")

    with gr.Row():
        with gr.Column():
            ui_text = gr.Textbox(label="Tokenizer Input", placeholder="Enter emotional/synchrony context...")
            ui_mode = gr.Radio(["Single", "Two-Peer"], label="Peer Mode", value="Two-Peer")
            ui_a = gr.Slider(0, 1, value=0.5, label="Peer A Telemetry")
            ui_b = gr.Slider(0, 1, value=0.5, label="Peer B Telemetry")
            run_btn = gr.Button("🚀 Encode & Execute Synthesis", variant="primary")

        with gr.Column():
            ui_output = gr.Markdown("### Waiting for input...")

    ui_mode.change(lambda m: gr.update(visible=(m == "Two-Peer")), inputs=ui_mode, outputs=ui_b)
    run_btn.click(fn=execute_quantum_synthesis, inputs=[ui_text, ui_mode, ui_a, ui_b], outputs=ui_output)

wanalytics_ui.launch(share=True)

WARNING    /tmp/ipython-input-226093401.py:25: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_ui:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft()) as wanalytics_ui:



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://313e14ece40659c26f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [60]:
import os, json, time, zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image

import librosa
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer

import cirq
import qsimcirq


# -------------------------
# 1) Load distilled student + norm (same as your notebook)
# -------------------------
MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

print("Loading text embedding model (SentenceTransformer)...")
embedder = SentenceTransformer("all-mpnet-base-v2")

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)


# -------------------------
# 2) Optional archive browsing (same idea as your notebook)
# -------------------------
EXTRACT_DIR = "./archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    return [str(p) for p in Path(root).rglob("*") if p.is_file()]

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)


# -------------------------
# 3) Feature extraction (compatible with your norm schema)
# -------------------------
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext == ".csv" else "\t"
        try:
            import pandas as pd
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n") + 1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s) / max(1, len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32) / 255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if (isinstance(path_or_text, str) and len(path_or_text) > 0 and not os.path.exists(path_or_text)) else load_text(path_or_text)
        feats.update(featurize_text(s))
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]

    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"

    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")

    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None

    else:
        preview = f"Unsupported modality for: {path_or_text}"

    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        x[i] = np.float32(feats.get(col, 0.0))
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y  # [valence, arousal, calm, trust]


# -------------------------
# 4) Quantum "synaptic" mapper (NEW)
# -------------------------
def build_synapse_circuit(e, n_qubits=4, depth=2, seed=0):
    """
    e: length-4 vector in [0,1] (valence, arousal, calm, trust)
    Encodes e into rotations, applies entangling layers, measures Z expectations.
    """
    assert len(e) == 4
    qs = cirq.LineQubit.range(n_qubits)

    # Map [0,1] -> angles
    angles = (np.clip(np.asarray(e, dtype=np.float32), 0, 1) * np.pi).tolist()

    circuit = cirq.Circuit()
    # input encoding
    for i in range(n_qubits):
        a = angles[i % 4]
        circuit.append(cirq.ry(a)(qs[i]))
        circuit.append(cirq.rz(0.5 * a)(qs[i]))

    # entangling "synapse" layers
    rng = np.random.default_rng(seed)
    for d in range(depth):
        # ring entanglement
        for i in range(n_qubits):
            circuit.append(cirq.CZ(qs[i], qs[(i+1) % n_qubits]))
        # local mixing
        for i in range(n_qubits):
            theta = float(rng.uniform(0.0, np.pi/2))
            circuit.append(cirq.rx(theta)(qs[i]))

    return circuit, qs

def synapse_expectations(circuit, qs):
    """
    Fast simulation with qsimcirq; return <Z_i> for each qubit.
    """
    sim = qsimcirq.QSimSimulator()
    result = sim.simulate(circuit)
    state = result.final_state_vector

    exps = []
    for i, q in enumerate(qs):
        # cirq.Z expectation
        exp = cirq.expectation_from_state_vector(
            state_vector=state,
            observable=cirq.Z(q),
            qubit_map={q: i},
        )
        exps.append(float(np.real(exp)))
    return exps  # each in [-1, 1]

def decode_haptics_from_exps(exps, base_hz=140.0):
    """
    Convert quantum expectations to haptic control values.
    """
    z = np.asarray(exps, dtype=np.float32)
    # [-1,1] -> [0,1]
    u = (z + 1.0) / 2.0

    intensity = float(np.clip(u.mean(), 0, 1))
    freq_hz = float(base_hz + 120.0 * (u[0] - 0.5) + 80.0 * (u[1] - 0.5))

    # simple pattern grammar
    if u[2] > 0.66:
        pattern = "pulse_train"
    elif u[3] > 0.66:
        pattern = "sweep_up"
    else:
        pattern = "steady"

    return {
        "intensity": intensity,
        "frequency_hz": max(10.0, freq_hz),
        "pattern": pattern,
        "quantum_exps": exps,
    }


# -------------------------
# 5) Session log (same spirit as your notebook)
# -------------------------
SESSION_LOG = []

def log_step(label, modality, source, y, feats, hapt=None):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()},
        "haptics": hapt or {},
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "./holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out


# -------------------------
# 6) Unified inference + quantum + UI
# -------------------------
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(str(Path(EXTRACT_DIR)) + os.sep, "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if not rel_path:
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text, use_quantum=True, depth=2):
    if modality == "text" and free_text and free_text.strip():
        x, feats, preview = make_feature_vector("text", free_text)
        src = "free_text"
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return None, "Pick a file (or paste text).", 0,0,0,0, {}, {}
        x, feats, preview = make_feature_vector(modality, path)
        src = rel_file

    y = predict_from_x(x)

    hapt = {}
    circ_txt = ""
    if use_quantum:
        circ, qs = build_synapse_circuit(y, n_qubits=4, depth=int(depth))
        exps = synapse_expectations(circ, qs)
        hapt = decode_haptics_from_exps(exps)
        circ_txt = str(circ)

    log_step("single", modality, src, y, feats, hapt=hapt)

    # preview routing
    preview_img = preview if isinstance(preview, Image.Image) else None
    preview_txt = "" if preview_img is not None else str(preview)

    return preview_img, preview_txt, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats, {"circuit": circ_txt, **hapt}

def infer_pair(modA, fileA, textA, modB, fileB, textB, use_quantum=True, depth=2):
    A = infer_one(modA, fileA, textA, use_quantum=use_quantum, depth=depth)
    B = infer_one(modB, fileB, textB, use_quantum=use_quantum, depth=depth)

    _, _, vA,aA,cA,tA, _, _ = A
    _, _, vB,aB,cB,tB, _, _ = B

    eA = np.array([vA,aA,cA,tA], dtype=np.float32)
    eB = np.array([vB,aB,cB,tB], dtype=np.float32)
    sync = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))
    return A[0], A[1], B[0], B[1], sync

with gr.Blocks(title="HoloSyn Unified (Classical + Quantum Synapse)") as demo:
    gr.Markdown("## HoloSyn Unified Interface — Distilled Student + Quantum Synaptic Mapper")

    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ Archive.zip not found. File browsing disabled; text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File (from Archive, optional)")
        free_text = gr.Textbox(lines=6, label="Text (used if modality=text)")

        with gr.Row():
            use_quantum = gr.Checkbox(value=True, label="Use quantum synapse mapper")
            depth = gr.Slider(1, 6, value=2, step=1, label="Quantum depth")

        run_btn = gr.Button("Run")

        with gr.Row():
            preview_img = gr.Image(label="Preview image/video frame", type="pil")
            preview_txt = gr.Textbox(label="Preview text", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)

        feats_json = gr.JSON(label="Extracted features")
        hapt_json = gr.JSON(label="Quantum→Haptics output (and circuit)")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        run_btn.click(
            fn=infer_one,
            inputs=[modality, file_dd, free_text, use_quantum, depth],
            outputs=[preview_img, preview_txt, val, aro, calm, trust, feats_json, hapt_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Compute synchrony on [valence, arousal, calm, trust].")
        with gr.Row():
            modA = gr.Dropdown(["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A")

        with gr.Row():
            modB = gr.Dropdown(["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B")

        with gr.Row():
            use_q2 = gr.Checkbox(value=True, label="Use quantum mapper for each side")
            depth2 = gr.Slider(1, 6, value=2, step=1, label="Quantum depth")

        run2 = gr.Button("Run A/B")
        with gr.Row():
            prevA_img = gr.Image(label="A preview (img)")
            prevA_txt = gr.Textbox(label="A preview (txt)", lines=6)
        with gr.Row():
            prevB_img = gr.Image(label="B preview (img)")
            prevB_txt = gr.Textbox(label="B preview (txt)", lines=6)

        sync = gr.Slider(-1, 1, value=0.0, step=0.001, label="Synchrony (cosine)", interactive=False)

        def refresh_filesA(m): return gr.update(choices=list_options(m), value=None)
        def refresh_filesB(m): return gr.update(choices=list_options(m), value=None)
        modA.change(refresh_filesA, inputs=[modA], outputs=[fileA])
        modB.change(refresh_filesB, inputs=[modB], outputs=[fileB])

        def run_pair(modA, fileA, textA, modB, fileB, textB, use_q, d):
            A_img, A_txt, *_ = infer_one(modA, fileA, textA, use_quantum=use_q, depth=d)
            B_img, B_txt, *_ = infer_one(modB, fileB, textB, use_quantum=use_q, depth=d)

            eA = np.array(predict_from_x(make_feature_vector("text", textA)[0]) if (modA=="text" and textA.strip()) else [0,0,0,0], dtype=np.float32)
            # Use outputs already computed in infer_one logs is possible, but keep simple:
            # We'll recompute synchrony using latest two SESSION_LOG entries:
            if len(SESSION_LOG) >= 2:
                rA, rB = SESSION_LOG[-2], SESSION_LOG[-1]
                vA,aA,cA,tA = rA["valence"], rA["arousal"], rA["calm"], rA["trust"]
                vB,aB,cB,tB = rB["valence"], rB["arousal"], rB["calm"], rB["trust"]
                EA = np.array([vA,aA,cA,tA], dtype=np.float32)
                EB = np.array([vB,aB,cB,tB], dtype=np.float32)
                s = float(np.dot(EA, EB) / (np.linalg.norm(EA)*np.linalg.norm(EB) + 1e-8))
            else:
                s = 0.0
            return A_img, A_txt, B_img, B_txt, s

        run2.click(
            fn=run_pair,
            inputs=[modA, fileA, textA, modB, fileB, textB, use_q2, depth2],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export"):
        gr.Markdown("Export the session log.")
        export_btn = gr.Button("Export JSON")
        export_out = gr.Textbox(label="Path")
        export_btn.click(fn=export_log, inputs=[], outputs=[export_out])

demo.launch()

Loading text embedding model (SentenceTransformer)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a647effca77cc5dabe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [71]:
# === ONE-CELL: HoloSyn Unified (Distilled Student + Quantum Synapse + Haptics UI) ===
# Uses your uploaded artifacts in /mnt/data
# - TorchScript student: /mnt/data/student_distilled_export.torchscript.pt (fallbacks below)
# - Norm schema:        /mnt/data/student_norm_hf.json

import os, json, time, math, warnings, subprocess, sys
from pathlib import Path

def _pip_install(pkgs):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
    except Exception as e:
        print("pip install failed:", e)
        raise

# --- deps (quiet install) ---
_pip_install([
    "gradio>=4.0.0",
    "numpy",
    "torch",
    "Pillow",
    "sentence-transformers",
    "librosa",
    "opencv-python-headless",
    "cirq",
    "qsimcirq",
])

import numpy as np
import torch
import gradio as gr
from PIL import Image

import librosa
import cv2

import cirq
import qsimcirq

from sentence_transformers import SentenceTransformer


# -------------------------
# 1) Locate your model files
# -------------------------
CANDIDATE_MODELS = [
    "./student_distilled_export.torchscript.pt",
    "./student_distilled_export.torchscript (1).pt",
    "./student_distilled_heads_hf.torchscript.pt",
    "./student_distilled_heads_hf.torchscript (1).pt",
    "./student_distilled_heads.torchscript.pt",
    "./holosyn_heads.torchscript.pt",
]
NORM_PATH = "./student_norm_hf.json"

MODEL_PATH = next((p for p in CANDIDATE_MODELS if os.path.exists(p)), None)
if MODEL_PATH is None:
    raise FileNotFoundError("Couldn't find a TorchScript model in /mnt/data. Checked:\n" + "\n".join(CANDIDATE_MODELS))
if not os.path.exists(NORM_PATH):
    raise FileNotFoundError(f"Missing norm file: {NORM_PATH}")

print("Using MODEL:", MODEL_PATH)
print("Using NORM :", NORM_PATH)

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)
SD[SD == 0] = 1.0

print("Feature dims:", len(NUMERIC_COLS))
print("First 12 cols:", NUMERIC_COLS[:12])


# -------------------------
# 2) Embedder (text -> 768 dims)
# -------------------------
# NOTE: This must match your schema which expects w2v_0..w2v_767 keys.
embedder = SentenceTransformer("all-mpnet-base-v2")


# -------------------------
# 3) Feature extraction compatible with your norm schema
# -------------------------
AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}

def _safe_float(x):
    try: return float(x)
    except Exception: return 0.0

def featurize_text(s: str):
    s = s or ""
    return {
        "txt_len": _safe_float(len(s)),
        "txt_lines": _safe_float(s.count("\n") + 1),
        "txt_exclaim": _safe_float(s.count("!")),
        "txt_question": _safe_float(s.count("?")),
        "txt_caps_ratio": _safe_float(sum(c.isupper() for c in s) / max(1, len(s))),
    }

def featurize_haptics_json(hobj):
    try:
        raw = json.dumps(hobj)[:20000].lower()
    except Exception:
        raw = str(hobj)[:20000].lower()
    return {
        "hapt_len": _safe_float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw or "frequency" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y is None or y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32) / 255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }, im

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (max(1,int(w*scale)), max(1,int(h*scale))))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, text_input=None, file_path=None, haptics_json=None):
    feats = {}
    preview_img = None
    preview_txt = ""

    if modality == "text":
        s = (text_input or "").strip()
        feats.update(featurize_text(s))
        emb = embedder.encode(s if s else " ")
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview_txt = s[:1500]

    elif modality == "haptics":
        try:
            hobj = json.loads(haptics_json) if isinstance(haptics_json, str) else (haptics_json or {})
        except Exception:
            hobj = {"raw": str(haptics_json)}
        feats.update(featurize_haptics_json(hobj))
        preview_txt = json.dumps(hobj)[:1500]

    elif modality == "audio":
        if not file_path: raise ValueError("Need an audio file.")
        feats.update(audio_features(file_path))
        preview_txt = f"Audio: {Path(file_path).name}"

    elif modality == "image":
        if not file_path: raise ValueError("Need an image file.")
        img_feats, pil = image_quick_stats(file_path)
        feats.update(img_feats)
        preview_img = pil

    elif modality == "video":
        if not file_path: raise ValueError("Need a video file.")
        frames = sample_video_frames(file_path)
        feats["vid_n_frames"] = float(len(frames))
        if frames:
            preview_img = Image.fromarray(frames[0]).convert("RGB")
        preview_txt = f"Video: {Path(file_path).name} (sampled_frames={len(frames)})"
    else:
        raise ValueError("Unknown modality.")

    # pack into x in NUMERIC_COLS order
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        x[i] = np.float32(feats.get(col, 0.0))
    return x, feats, preview_img, preview_txt


# -------------------------
# 4) Student inference
# -------------------------
def student_predict(x: np.ndarray):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt)
    if isinstance(y, (tuple, list)):
        y = y[0]
    y = y.squeeze(0).detach().cpu().numpy().astype(np.float32).ravel()
    # Expect 4 dims: [valence, arousal, calm, trust]; if longer, take first 4.
    if y.size < 4:
        y = np.pad(y, (0, 4 - y.size), mode="constant")
    return y[:4]


# -------------------------
# 5) Quantum synapse layer (Cirq + qsimcirq)
# -------------------------
def build_synapse_circuit(state4, n_qubits=4, depth=2, seed=0):
    qs = cirq.LineQubit.range(n_qubits)
    e = np.clip(np.asarray(state4, dtype=np.float32), 0.0, 1.0)
    angles = (e * np.pi).tolist()

    circuit = cirq.Circuit()
    for i in range(n_qubits):
        a = angles[i % 4]
        circuit.append(cirq.ry(a)(qs[i]))
        circuit.append(cirq.rz(0.5 * a)(qs[i]))

    rng = np.random.default_rng(seed)
    for _ in range(int(depth)):
        for i in range(n_qubits):
            circuit.append(cirq.CZ(qs[i], qs[(i+1) % n_qubits]))
        for i in range(n_qubits):
            circuit.append(cirq.rx(float(rng.uniform(0.0, np.pi/2)))(qs[i]))

    return circuit, qs

def synapse_expectations(circuit, qs):
    sim = qsimcirq.QSimSimulator()
    result = sim.simulate(circuit)
    state = result.final_state_vector

    exps = []
    for i, q in enumerate(qs):
        exp = cirq.expectation_from_state_vector(
            state_vector=state,
            observable=cirq.Z(q),
            qubit_map={q: i},
        )
        exps.append(float(np.real(exp)))
    return exps  # [-1,1]

def decode_haptics_from_exps(exps, base_hz=140.0):
    z = np.asarray(exps, dtype=np.float32)
    u = (z + 1.0) / 2.0  # -> [0,1]
    intensity = float(np.clip(u.mean(), 0, 1))
    frequency_hz = float(base_hz + 120.0*(u[0]-0.5) + 80.0*(u[1]-0.5))
    if u[2] > 0.66:
        pattern = "pulse_train"
    elif u[3] > 0.66:
        pattern = "sweep_up"
    else:
        pattern = "steady"
    return {
        "intensity": intensity,
        "frequency_hz": max(10.0, frequency_hz),
        "pattern": pattern,
        "quantum_exps": exps,
    }


# -------------------------
# 6) Unified run function (UI callback)
# -------------------------
SESSION_LOG = []

def run_unified(modality, text_input, file_input, haptics_json, use_quantum, q_depth):
    try:
        # gradio provides file_input as dict or str depending on version
        file_path = None
        if file_input:
            if isinstance(file_input, str):
                file_path = file_input
            elif isinstance(file_input, dict) and "name" in file_input:
                file_path = file_input["name"]

        x, feats, prev_img, prev_txt = make_feature_vector(
            modality=modality,
            text_input=text_input,
            file_path=file_path,
            haptics_json=haptics_json
        )
        y = student_predict(x)
        out = {
            "valence": float(y[0]),
            "arousal": float(y[1]),
            "calm": float(y[2]),
            "trust": float(y[3]),
        }

        quantum = {}
        if use_quantum:
            circ, qs = build_synapse_circuit(y, n_qubits=4, depth=int(q_depth))
            exps = synapse_expectations(circ, qs)
            quantum = decode_haptics_from_exps(exps)
            quantum["circuit"] = str(circ)

        rec = {
            "ts": time.time(),
            "modality": modality,
            "valence": out["valence"],
            "arousal": out["arousal"],
            "calm": out["calm"],
            "trust": out["trust"],
            "quantum": quantum,
        }
        SESSION_LOG.append(rec)

        return (
            prev_img,
            prev_txt,
            out["valence"], out["arousal"], out["calm"], out["trust"],
            feats,
            quantum
        )
    except Exception as e:
        return None, f"Error: {e}", 0,0,0,0, {}, {}


# -------------------------
# 7) Gradio unified interface
# -------------------------
with gr.Blocks(title="HoloSyn Unified (Distilled + Quantum Synapse)") as demo:
    gr.Markdown("## HoloSyn Unified — Distilled Student + Quantum Synaptic Haptics (one-cell build)")

    with gr.Row():
        modality = gr.Dropdown(["text", "image", "audio", "video", "haptics"], value="text", label="Modality")
        use_quantum = gr.Checkbox(value=True, label="Enable quantum synapse (Cirq/qsimcirq)")
        q_depth = gr.Slider(1, 6, value=2, step=1, label="Quantum depth")

    text_input = gr.Textbox(lines=6, label="Text (used for modality=text)")
    file_input = gr.File(label="File (used for image/audio/video)")
    haptics_json = gr.Textbox(lines=6, label="Haptics JSON (used for modality=haptics)", value='{"intensity":0.5,"frequency_hz":120,"pattern":"steady"}')

    run_btn = gr.Button("Run")

    with gr.Row():
        preview_img = gr.Image(label="Preview (image/video frame)", type="pil")
        preview_txt = gr.Textbox(label="Preview (text/audio/haptics)", lines=10)

    with gr.Row():
        val = gr.Slider(0, 1, step=0.001, label="Valence", interactive=False)
        aro = gr.Slider(0, 1, step=0.001, label="Arousal", interactive=False)
        calm = gr.Slider(0, 1, step=0.001, label="Calm", interactive=False)
        trust = gr.Slider(0, 1, step=0.001, label="Trust", interactive=False)

    feats_json = gr.JSON(label="Extracted features (sparse dict)")
    quantum_json = gr.JSON(label="Quantum→Haptics output (and circuit)")

    run_btn.click(
        fn=run_unified,
        inputs=[modality, text_input, file_input, haptics_json, use_quantum, q_depth],
        outputs=[preview_img, preview_txt, val, aro, calm, trust, feats_json, quantum_json],
    )

demo.launch()

Using MODEL: ./student_distilled_export.torchscript.pt
Using NORM : ./student_norm_hf.json
Feature dims: 789
First 12 cols: ['txt_len', 'txt_lines', 'txt_exclaim', 'txt_question', 'txt_caps_ratio', 'hapt_len', 'hapt_has_intensity', 'hapt_has_freq', 'aud_rms', 'aud_zcr', 'aud_centroid', 'aud_tempo']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3616f8a32c6fd68e78.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
